# Field Aliasing & Serialization

In real-world applications, the data structures coming into your system (JSON, APIs, databases) often use different naming conventions than your internal Python code. **Field Aliasing** bridges this gap, allowing you to use Pythonic variable names internally while accepting and returning data in whatever format the outside world expects.

## 1. Why Do We Need Aliases?
There are two primary reasons we use aliases:
1. **Naming Conventions:** Python strictly uses `snake_case` (e.g., `first_name`). JSON and JavaScript strictly use `camelCase` (e.g., `firstName`). Aliases allow you to accept `firstName` from a JSON payload but interact with it as `model.first_name` in Python.
2. **Reserved Keywords:** A JSON payload might send a key called `id`. In Python, `id` is a built-in function, so using it as a variable name is heavily frowned upon by linters. We can name our Python field `id_` and use an alias to map it to the JSON `id`.

## 2. The Three Types of Aliases
Pydantic uses the `Field` function to define aliases. There are three specific types of aliases you can configure:

* **`alias` (The Standard Alias):** This is a two-way street. It is used for *both* deserialization (reading input) and serialization (generating output).
* **`validation_alias`:** This is a one-way street for **Input Only**. Pydantic looks for this key when instantiating the model.
* **`serialization_alias`:** This is a one-way street for **Output Only**. Pydantic uses this key when dumping the model to a dictionary or JSON.

*Note: If you define both a `validation_alias` and a `serialization_alias`, Pydantic will completely ignore the standard `alias`.*

```python
from pydantic import BaseModel, Field

class User(BaseModel):
    # Standard Alias (Input and Output)
    last_name: str = Field(alias="lastName")
    
    # Split Aliases
    # Looks for 'id' in JSON, outputs 'user_id' in JSON, but is called 'id_' in Python
    id_: int = Field(validation_alias="id", serialization_alias="user_id")

```

## 3. Deserializing & Serializing with Aliases

### A. Deserialization (`populate_by_name`)

By default, if you define an alias, Pydantic **forces** you to use that alias when creating the object. It will reject the actual Python field name. To allow *either* the field name or the alias during creation, use `populate_by_name`.

```python
from pydantic import ConfigDict

class Profile(BaseModel):
    model_config = ConfigDict(populate_by_name=True) # Enables using either name
    
    first_name: str = Field(alias="firstName")

# Both of these now work!
p1 = Profile(firstName="John")
p2 = Profile(first_name="John")

```

### B. Serialization (`by_alias`)

When you serialize a model (convert it to a dictionary or JSON), Pydantic defaults to using your Python field names. If you want it to output your aliases, you must explicitly pass `by_alias=True` to the dump method.

```python
# Output using Python field names (Default)
print(p1.model_dump()) 
# {'first_name': 'John'}

# Output using the defined aliases
print(p1.model_dump(by_alias=True)) 
# {'firstName': 'John'}

```

## 4. Multiple Validation Aliases (`AliasChoices`)

Sometimes you receive data from multiple different sources (e.g., a modern API and a legacy database), and the key for the same data might be named differently depending on the source. Pydantic provides `AliasChoices` to check multiple keys during deserialization.

```python
from pydantic import AliasChoices

class Employee(BaseModel):
    # Will check the input data for 'first_name', then 'fname', then 'firstName'
    first_name: str = Field(validation_alias=AliasChoices('first_name', 'fname', 'firstName'))

e1 = Employee.model_validate({'fname': 'Alice'})
print(e1.first_name) # 'Alice'

```

## 5. Auto-Generating Aliases (`alias_generator`)

If you have a model with 50 fields, writing `Field(alias="...")` 50 times is tedious. Pydantic allows you to specify a function at the `model_config` level that automatically generates aliases for every field.

Pydantic even provides built-in generators for common conversions like `to_camel`.

```python
from pydantic.alias_generators import to_camel

class AutoCamelModel(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel, 
        populate_by_name=True
    )
    
    first_name: str
    last_name: str
    
    # You can still manually override the generator for specific fields!
    id_: int = Field(alias="id")

# The generator automatically expects 'firstName' and 'lastName'
data = {"id": 1, "firstName": "Bob", "lastName": "Smith"}
m = AutoCamelModel(**data)

```

### Interview Preparation: Field Aliasing

<details>
<summary><b>1. Why should you use Pydantic aliases instead of just naming your Python variables to match the incoming JSON data?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
There are two main reasons: maintaining language conventions and avoiding reserved keywords. <br><br>
JSON APIs universally use `camelCase` (e.g., `userId`), whereas Python strictly adheres to PEP 8 standard `snake_case` (e.g., `user_id`). Using aliases allows you to maintain clean, idiomatic Python code internally while perfectly interfacing with standard JSON APIs. Secondly, JSON often uses keys like `id`, `type`, or `class`, which are reserved keywords or built-in functions in Python. Aliasing allows you to map a JSON `id` to a safe Python variable like `id_`.
</details>

<details>
<summary><b>2. You define a field as `first_name: str = Field(alias="firstName")`. When you attempt to instantiate the model using `User(first_name="Alice")`, Pydantic throws a ValidationError. Why did this happen and how do you fix it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
By default, when you define an alias in Pydantic, it strictly expects the alias to be used during instantiation and hides the underlying field name from the initializer. <br><br>
To fix this, you need to add `model_config = ConfigDict(populate_by_name=True)` to your model. This configuration tells Pydantic to accept *either* the alias (`firstName`) or the actual Python field name (`first_name`) when deserializing data.
</details>

<details>
<summary><b>3. Explain the difference between `validation_alias` and `serialization_alias`.</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
A standard `alias` applies to both the input and the output. However, sometimes you need asymmetrical data flows. <br><br>
A `validation_alias` is used strictly during <b>deserialization</b> (input). It tells Pydantic what key to look for when reading incoming data. <br><br>
A `serialization_alias` is used strictly during <b>serialization</b> (output). It tells Pydantic what key to output when you call `model_dump(by_alias=True)`. <br><br>
For example, you might use a `validation_alias` to read a legacy database column `usr_nm`, but use a `serialization_alias` to output it to a modern frontend as `userName`.
</details>

<details>
<summary><b>4. You are writing an API that needs to support both an old v1 payload (`{"emp_id": 123}`) and a new v2 payload (`{"employeeId": 123}`). How do you map both of these incoming keys to a single `id` field in your Pydantic model?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You would use Pydantic's `AliasChoices` class inside a `validation_alias`. <br><br>
I would define the field as: `id: int = Field(validation_alias=AliasChoices('employeeId', 'emp_id'))`. <br>
When Pydantic deserializes the payload, it will check the data for `employeeId` first. If it doesn't find it, it will fall back and check for `emp_id`. If either is present, the data successfully maps to the `id` field.
</details>

# Implementing Aliases with the `Field` Object

Up until now, we have configured fields simply by using type hints and default values (e.g., `age: int = 18`). However, Pydantic provides a much more powerful tool for per-field configuration: the `Field` object. 

In this section, we will use the `Field` object to implement the **Field Aliasing** concepts discussed previously, allowing us to map external data formats (like JSON's `camelCase`) to internal Python standards (like `snake_case`).

## 1. Defining Aliases with `Field`
To attach an alias to a field, we assign an instance of the `Field` object as the field's value, passing the `alias` argument.

```python
from pydantic import BaseModel, Field, ValidationError

class User(BaseModel):
    # Mapping JSON 'id' to a safe Python variable 'id_'
    id_: int = Field(alias="id")
    
    # Mapping JSON 'lastName' to Python 'last_name'
    last_name: str = Field(alias="lastName")

# 1. Deserialization MUST use the aliases by default
json_payload = '{"id": 100, "lastName": "Gauss"}'
u1 = User.model_validate_json(json_payload)

# Using keyword arguments also requires the aliases!
u2 = User(id=101, lastName="Newton")

```

### The Deserialization Trap

By default, once you define an alias, Pydantic **strictly expects the alias** during deserialization. If you try to instantiate the model using the actual Python field names, Pydantic will not recognize them and will throw a missing field `ValidationError`.

```python
try:
    # ❌ Fails because it expects 'id' and 'lastName'
    User(id_=102, last_name="Einstein") 
except ValidationError as e:
    print("Error: Field required")

```

## 2. Internal Access

Aliases are strictly for crossing the boundary (serialization and deserialization). Once the data is successfully loaded into the Python object, **you completely drop the alias and use the Python field names**.

```python
# ✅ Use the Pythonic names in your codebase
print(u1.id_)       # 100
print(u1.last_name) # 'Gauss'

# ❌ The alias does not exist as an attribute
# print(u1.lastName) -> AttributeError

```

## 3. Setting Defaults inside `Field`

Because you are now using the right side of the equals sign to instantiate the `Field` object, you can no longer define default values using standard Python assignment. You must move your default value inside the `Field` object using the `default` parameter.

```python
class OptionalUser(BaseModel):
    # Defining an alias AND a default value
    id_: int = Field(alias="id", default=0)
    
    # Nullable, optional, and aliased
    first_name: str | None = Field(alias="firstName", default=None)

u3 = OptionalUser()
print(u3.id_) # 0

```

## 4. Serialization: Outputting Aliases

When you are ready to send your data back out to the world (e.g., returning a JSON response from an API), Pydantic defaults to outputting your internal Python field names.

If you want to output the aliases (to satisfy your API contract), you must explicitly tell Pydantic to do so by passing `by_alias=True` to the dump methods.

```python
# Default Serialization (Uses Python names)
print(u1.model_dump()) 
# {'id_': 100, 'last_name': 'Gauss'}

# Serialization by Alias (Uses the external names)
print(u1.model_dump(by_alias=True)) 
# {'id': 100, 'lastName': 'Gauss'}

# This works for JSON generation as well
print(u1.model_dump_json(by_alias=True)) 
# '{"id":100,"lastName":"Gauss"}'

```

### Interview Preparation: The Field Object & Aliases

<details>
<summary><b>1. You mapped a JSON key `"id"` to the Python field `id_` using `id_: int = Field(alias="id")`. How do you access this value on the resulting model instance `user`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You access it using `user.id_`. <br><br>
Aliases are only used at the boundaries of your application (deserialization and serialization). Inside your Python codebase, the object behaves like a standard Python class, and you access the data using the defined Python attribute names. Attempting to call `user.id` will result in an `AttributeError`.
</details>

<details>
<summary><b>2. You defined a field as `age: int = Field(alias="userAge")`. You want the field to be optional with a default of `18`. If you write `age: int = Field(alias="userAge") = 18`, Python throws a SyntaxError. How do you correctly assign the default value?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Because you are replacing the standard assignment with a `Field` configuration object, you must pass the default value as an argument inside the `Field` constructor. <br><br>
The correct syntax is: `age: int = Field(alias="userAge", default=18)`.
</details>

<details>
<summary><b>3. Your Pydantic model uses aliases to map Python's `snake_case` to JSON's `camelCase`. When you return `model.model_dump_json()` in your API response, the frontend developer complains that the JSON is formatted in `snake_case`. Why did this happen and how do you fix it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
By default, Pydantic's serialization methods (`model_dump` and `model_dump_json`) output the internal Python field names, ignoring the aliases entirely. <br><br>
To fix this and output the `camelCase` aliases as required by the frontend, you must explicitly pass the `by_alias` parameter to the serialization method: `model.model_dump_json(by_alias=True)`.
</details>

<details>
<summary><b>4. You define a model with `last_name: str = Field(alias="lastName")`. A developer writes a test case and instantiates the model using `User(last_name="Smith")`. The test fails with a ValidationError stating that `lastName` is required. Explain why.</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
By default, when you define an alias on a field, Pydantic strictly enforces the use of that alias during object creation (deserialization), hiding the underlying Python field name from the constructor. <br><br>
Because the developer passed the Python field name (`last_name`), Pydantic rejected it and threw an error looking for the `lastName` alias. To allow both, the model must be configured with `model_config = ConfigDict(populate_by_name=True)`.
</details>